# VQ-VAE Codebook & Class Analysis

Analyzes how codebook usage patterns correlate with diagnostic classes (AD, CN, MCI).
Includes PCA/t-SNE visualizations of both codebook histograms and continuous encoder features.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import chi2_contingency
from tqdm.auto import tqdm

from eval import load_model_from_checkpoint, get_transforms
from utils import load_items
from monai.data import Dataset

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Configuration

Set paths below before running.

In [ ]:
CHECKPOINT_PATH = "/home/ng24/projects/vqvae-smm/results/checkpoint_best.pt"  # TODO: fill in
CSV_PATH = "/home/ng24/projects/nmpevqvae/labels_cleaned_3class.csv"
DATAROOT = "/data/natalia/ADNI_registered"

SPACING = 2.0        # voxel spacing in mm (2.0 = faster, 1.0 = full res)
BATCH_SIZE = 4
NUM_WORKERS = 4
MAX_SUBJECTS = None  # set to e.g. 100 to cap dataset size (None = use all)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_NAMES = ["AD", "CN", "MCI"]  # sorted order matches label_map in load_items
CLASS_COLORS = {"AD": "#e74c3c", "CN": "#2ecc71", "MCI": "#3498db"}

print(f"Device: {DEVICE}")

## 2. Load Model & Data

In [ ]:
model = load_model_from_checkpoint(CHECKPOINT_PATH, device=DEVICE)
model.eval()

nb_levels = model.nb_levels
nb_entries = model.codebooks[0].n_embed
print(f"Model: {nb_levels} levels, {nb_entries} codebook entries each")

In [ ]:
items = load_items(DATAROOT, CSV_PATH)

# Optionally subsample to avoid OOM
if MAX_SUBJECTS is not None and len(items) > MAX_SUBJECTS:
    rng = np.random.RandomState(42)
    idx = rng.choice(len(items), MAX_SUBJECTS, replace=False)
    idx.sort()
    items = [items[i] for i in idx]

print(f"Using {len(items)} subjects")

transforms = get_transforms(spacing=SPACING)
dataset = Dataset(
    data=[{"image": it["image"]} for it in items],
    transform=transforms,
)

labels = np.array([it["label"] for it in items])
subjects = [it["subject"] for it in items]

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f"Class distribution: {dict(zip(CLASS_NAMES, np.bincount(labels)))}")

## 3. Extract Codebook Indices & Continuous Features

In [ ]:
# Storage: per-level lists
all_histograms = [[] for _ in range(nb_levels)]   # codebook usage histograms
all_pooled = [[] for _ in range(nb_levels)]        # continuous pooled encoder features

with torch.no_grad():
    for batch in tqdm(loader, desc="Extracting features"):
        images = batch["image"].to(DEVICE)
        _, diffs, encoder_pools, _, id_outputs = model(
            images, return_recon=False, pool_only=True
        )

        B = images.shape[0]
        for lvl in range(nb_levels):
            # Codebook index histograms
            ids = id_outputs[lvl]  # (B, D, H, W)
            for b in range(B):
                hist = torch.bincount(ids[b].reshape(-1), minlength=nb_entries)
                hist = hist.float() / hist.sum()  # normalize to frequency
                all_histograms[lvl].append(hist.cpu().numpy())

            # Pooled encoder features
            all_pooled[lvl].append(encoder_pools[lvl].cpu().numpy())

# Stack into arrays
for lvl in range(nb_levels):
    all_histograms[lvl] = np.array(all_histograms[lvl])  # (N, nb_entries)
    all_pooled[lvl] = np.concatenate(all_pooled[lvl], axis=0)  # (N, C)

N = all_histograms[0].shape[0]
print(f"Extracted features for {N} subjects")
for lvl in range(nb_levels):
    print(f"  Level {lvl}: histograms {all_histograms[lvl].shape}, pooled {all_pooled[lvl].shape}")

## 4. Codebook Usage by Class

In [ ]:
for lvl in range(nb_levels):
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    fig.suptitle(f"Level {lvl} — Codebook Usage by Class", fontsize=14)

    # --- Mean usage histogram per class ---
    ax = axes[0]
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        mask = labels == cls_idx
        mean_usage = all_histograms[lvl][mask].mean(axis=0)
        ax.bar(
            np.arange(nb_entries), mean_usage, alpha=0.5,
            label=cls_name, color=CLASS_COLORS[cls_name],
        )
    ax.set_xlabel("Codebook entry")
    ax.set_ylabel("Mean frequency")
    ax.set_title("Mean codebook usage")
    ax.legend()

    # --- Heatmap: classes x entries ---
    ax = axes[1]
    usage_matrix = np.zeros((len(CLASS_NAMES), nb_entries))
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        usage_matrix[cls_idx] = all_histograms[lvl][labels == cls_idx].mean(axis=0)
    sns.heatmap(
        usage_matrix, ax=ax, cmap="viridis",
        yticklabels=CLASS_NAMES, xticklabels=False,
    )
    ax.set_xlabel("Codebook entry")
    ax.set_title("Usage heatmap (class x entry)")

    plt.tight_layout()
    plt.show()

In [ ]:
TOP_N = 10  # number of top codes to show per class

for lvl in range(nb_levels):
    print(f"{'=' * 70}")
    print(f"LEVEL {lvl}")
    print(f"{'=' * 70}")

    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        mask = labels == cls_idx
        mean_usage = all_histograms[lvl][mask].mean(axis=0)
        top_indices = np.argsort(mean_usage)[::-1][:TOP_N]

        print(f"\n  {cls_name} (n={mask.sum()}) — top {TOP_N} most-used codes:")
        print(f"  {'Code':>6s}  {'Freq':>8s}  {'Bar'}")
        print(f"  {'─' * 6}  {'─' * 8}  {'─' * 30}")
        for idx in top_indices:
            freq = mean_usage[idx]
            bar = "█" * int(freq * 500)  # scale for display
            print(f"  {idx:6d}  {freq:8.4f}  {bar}")

    # Also show codes with biggest difference between classes
    usage_per_class = np.zeros((len(CLASS_NAMES), nb_entries))
    for cls_idx in range(len(CLASS_NAMES)):
        usage_per_class[cls_idx] = all_histograms[lvl][labels == cls_idx].mean(axis=0)

    # Which class uses each code the most?
    dominant_class = np.argmax(usage_per_class, axis=0)
    max_diff = usage_per_class.max(axis=0) - usage_per_class.min(axis=0)
    top_diff = np.argsort(max_diff)[::-1][:TOP_N]

    print(f"\n  Codes with largest cross-class difference:")
    print(f"  {'Code':>6s}  {'Dominant':>8s}  {'MaxFreq':>8s}  {'MinFreq':>8s}  {'Diff':>8s}")
    print(f"  {'─' * 6}  {'─' * 8}  {'─' * 8}  {'─' * 8}  {'─' * 8}")
    for idx in top_diff:
        dom = CLASS_NAMES[dominant_class[idx]]
        mx = usage_per_class[:, idx].max()
        mn = usage_per_class[:, idx].min()
        print(f"  {idx:6d}  {dom:>8s}  {mx:8.4f}  {mn:8.4f}  {mx - mn:8.4f}")
    print()

## 5. Most Discriminative Codes (Chi-squared)

In [ ]:
TOP_K = 20

for lvl in range(nb_levels):
    chi2_scores = np.zeros(nb_entries)
    for code_idx in range(nb_entries):
        # Bin usage into quartiles for chi-squared
        usage = all_histograms[lvl][:, code_idx]
        bins = np.quantile(usage[usage > 0], [0.33, 0.66]) if (usage > 0).sum() > 10 else None
        if bins is None or len(np.unique(bins)) < 2:
            continue
        digitized = np.digitize(usage, bins)
        contingency = pd.crosstab(digitized, labels)
        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            chi2, p, _, _ = chi2_contingency(contingency)
            chi2_scores[code_idx] = chi2

    top_codes = np.argsort(chi2_scores)[::-1][:TOP_K]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(range(TOP_K), chi2_scores[top_codes], color="steelblue")
    ax.set_xticks(range(TOP_K))
    ax.set_xticklabels(top_codes, rotation=45)
    ax.set_xlabel("Codebook entry index")
    ax.set_ylabel("Chi-squared statistic")
    ax.set_title(f"Level {lvl} — Top {TOP_K} most class-discriminative codes")
    plt.tight_layout()
    plt.show()

## 6. Mutual Information: Codebook Entry vs Class

In [ ]:
for lvl in range(nb_levels):
    mi_scores = mutual_info_classif(
        all_histograms[lvl], labels, discrete_features=False, random_state=42,
    )

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(np.arange(nb_entries), mi_scores, color="darkorange", width=1.0)
    ax.set_xlabel("Codebook entry")
    ax.set_ylabel("Mutual information (nats)")
    ax.set_title(f"Level {lvl} — MI between codebook entry usage and class label")
    plt.tight_layout()
    plt.show()

    top10 = np.argsort(mi_scores)[::-1][:10]
    print(f"Level {lvl} top-10 MI codes: {top10.tolist()}")
    print(f"  MI values: {mi_scores[top10].round(4).tolist()}")

## 7. PCA & t-SNE of Codebook Usage Histograms

In [ ]:
def scatter_by_class(ax, coords, labels, class_names, class_colors, title):
    for cls_idx, cls_name in enumerate(class_names):
        mask = labels == cls_idx
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            c=class_colors[cls_name], label=cls_name,
            alpha=0.6, s=15, edgecolors="none",
        )
    ax.legend(markerscale=2)
    ax.set_title(title)


for lvl in range(nb_levels):
    X = all_histograms[lvl]

    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X)

    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    X_tsne = tsne.fit_transform(X)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Level {lvl} — Codebook Histogram Embeddings", fontsize=14)

    scatter_by_class(
        axes[0], X_pca, labels, CLASS_NAMES, CLASS_COLORS,
        f"PCA (var explained: {pca.explained_variance_ratio_.sum():.1%})",
    )
    axes[0].set_xlabel("PC1")
    axes[0].set_ylabel("PC2")

    scatter_by_class(
        axes[1], X_tsne, labels, CLASS_NAMES, CLASS_COLORS, "t-SNE",
    )
    axes[1].set_xlabel("t-SNE 1")
    axes[1].set_ylabel("t-SNE 2")

    plt.tight_layout()
    plt.show()

## 8. PCA & t-SNE of Continuous Encoder Features

In [ ]:
for lvl in range(nb_levels):
    X = all_pooled[lvl]

    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X)

    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    X_tsne = tsne.fit_transform(X)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Level {lvl} — Continuous Encoder Features", fontsize=14)

    scatter_by_class(
        axes[0], X_pca, labels, CLASS_NAMES, CLASS_COLORS,
        f"PCA (var explained: {pca.explained_variance_ratio_.sum():.1%})",
    )
    axes[0].set_xlabel("PC1")
    axes[0].set_ylabel("PC2")

    scatter_by_class(
        axes[1], X_tsne, labels, CLASS_NAMES, CLASS_COLORS, "t-SNE",
    )
    axes[1].set_xlabel("t-SNE 1")
    axes[1].set_ylabel("t-SNE 2")

    plt.tight_layout()
    plt.show()

## 9. Combined Multi-Level Feature Analysis

Concatenate features across all levels for a joint view.

In [ ]:
# Concatenate histograms across levels
X_hist_all = np.concatenate(all_histograms, axis=1)
X_pool_all = np.concatenate(all_pooled, axis=1)

for name, X in [("Codebook histograms (all levels)", X_hist_all),
                ("Pooled features (all levels)", X_pool_all)]:
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X)
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    X_tsne = tsne.fit_transform(X)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(name, fontsize=14)

    scatter_by_class(
        axes[0], X_pca, labels, CLASS_NAMES, CLASS_COLORS,
        f"PCA (var explained: {pca.explained_variance_ratio_.sum():.1%})",
    )
    scatter_by_class(axes[1], X_tsne, labels, CLASS_NAMES, CLASS_COLORS, "t-SNE")

    plt.tight_layout()
    plt.show()

## 10. Codebook Vector Replacement & Reconstruction

Replace a specific codebook entry at a given level with another entry, decode
back to an image, and compare with the original reconstruction.

In [ ]:
import torch.nn.functional as F


@torch.no_grad()
def decode_from_indices(model, code_indices):
    """Decode a list of codebook index tensors back to an image (3D-safe).

    Args:
        model: VQVAE model.
        code_indices: list of LongTensors, one per level.
            Level ordering must match model convention (level 0 = finest).
            Each tensor has shape (B, D_l, H_l, W_l).

    Returns:
        Reconstructed image tensor (B, C, D, H, W).
    """
    decoder_outputs = []
    code_outputs = []
    upscale_counts = []

    for l in range(model.nb_levels - 1, -1, -1):
        codebook = model.codebooks[l]
        decoder = model.decoders[l]

        # embed_code -> (B, D, H, W, embed_dim), permute to (B, embed_dim, D, H, W)
        code_q = codebook.embed_code(code_indices[l]).permute(0, 4, 1, 2, 3)

        # Upscale previous code outputs
        upscaled_codes = []
        target_size = code_q.shape[2:]
        for i, c in enumerate(code_outputs):
            upscaled = model.upscalers[i](c, upscale_counts[i])
            if upscaled.shape[2:] != target_size:
                upscaled = F.interpolate(
                    upscaled, size=target_size, mode="trilinear", align_corners=False,
                )
            upscaled_codes.append(upscaled)
        code_outputs = upscaled_codes
        upscale_counts = [u + 1 for u in upscale_counts]

        decoder_in = torch.cat([code_q, *code_outputs], dim=1)
        decoder_outputs.append(decoder(decoder_in))

        code_outputs.append(code_q)
        upscale_counts.append(0)

    return decoder_outputs[-1]

In [ ]:
# ── Configuration ────────────────────────────────────────────────
SUBJECT_IDX = 0            # index into `items` list
TARGET_LEVEL = 2           # codebook level to modify (0=finest, nb_levels-1=coarsest)
OLD_CODE = 10              # codebook entry index to replace
NEW_CODE = 50              # replacement codebook entry index
SLICE_AXIS = 2             # 0=sagittal, 1=coronal, 2=axial

# ── Encode the subject ──────────────────────────────────────────
sample = dataset[SUBJECT_IDX]
image = sample["image"].unsqueeze(0).to(DEVICE)  # (1, C, D, H, W)

with torch.no_grad():
    recon_orig, _, _, _, id_outputs_raw = model(image, return_recon=True)

# id_outputs from the forward pass is coarsest-first (appended during the
# range(nb_levels-1, ..., -1) loop).  Reverse so index 0 = level 0 (finest).
id_outputs = id_outputs_raw[::-1]

# ── Replace codes ───────────────────────────────────────────────
modified_ids = [ids.clone() for ids in id_outputs]
n_replaced = (modified_ids[TARGET_LEVEL] == OLD_CODE).sum().item()
modified_ids[TARGET_LEVEL][modified_ids[TARGET_LEVEL] == OLD_CODE] = NEW_CODE
print(f"Level {TARGET_LEVEL}: replaced {n_replaced} voxels from code {OLD_CODE} → {NEW_CODE}")
print(f"  Code map shapes: {[tuple(ids.shape) for ids in id_outputs]}")

# ── Decode original & modified ──────────────────────────────────
with torch.no_grad():
    recon_orig_from_codes = decode_from_indices(model, id_outputs)
    recon_modified = decode_from_indices(model, modified_ids)

# Interpolate to match input spatial size if needed
input_shape = image.shape[2:]
if recon_orig_from_codes.shape[2:] != input_shape:
    recon_orig_from_codes = F.interpolate(
        recon_orig_from_codes, size=input_shape, mode="trilinear", align_corners=False,
    )
if recon_modified.shape[2:] != input_shape:
    recon_modified = F.interpolate(
        recon_modified, size=input_shape, mode="trilinear", align_corners=False,
    )

# ── Visualize ───────────────────────────────────────────────────
def get_mid_slice(vol, axis):
    """Get the middle slice along a given axis from a (C, D, H, W) tensor."""
    idx = vol.shape[axis + 1] // 2  # +1 to skip channel dim
    # also sample 2x quarter slices on either side for better visibility of differences
    idx1 = vol.shape[axis + 1] // 4
    idx3 = 3 * vol.shape[axis + 1] // 4
    return vol[0].select(axis, idx).cpu().numpy(), vol[0].select(axis, idx1).cpu().numpy(), vol[0].select(axis, idx3).cpu().numpy()

diff = (recon_modified - recon_orig_from_codes).abs()

fig, axes = plt.subplots(3, 4, figsize=(20, 5))
titles = ["Original input", "Decoded (original codes)", "Decoded (modified codes)", "Absolute difference"]
volumes = [image, recon_orig_from_codes, recon_modified, diff]

for i, (vol, title) in enumerate(zip(volumes, titles)):
    for j in range(3):
        slc, slc1, slc3 = get_mid_slice(vol[0], SLICE_AXIS)
        cmap = "hot" if "difference" in title.lower() else "gray"
        axes[j, i].imshow(slc, cmap=cmap, origin="lower")
        axes[j, i].set_title(title + (f" (slice {['mid', 'Q1', 'Q3'][j]})" if j > 0 else ""))
        axes[j, i].axis("off")

subject_name = subjects[SUBJECT_IDX] if SUBJECT_IDX < len(subjects) else f"#{SUBJECT_IDX}"
fig.suptitle(
    f"Subject {subject_name} — Level {TARGET_LEVEL}, code {OLD_CODE} → {NEW_CODE} "
    f"({n_replaced} voxels replaced)",
    fontsize=13,
)
plt.tight_layout()
plt.show()